#### Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

#### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [ ]:
import os
from langchain.chat_models import init_chat_model

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
model = init_chat_model("gpt-5-nano")
model

In [2]:
from pydantic import Field, BaseModel

class Movie(BaseModel):
    title: str=Field(description="Title of the movie")
    year: int=Field(description="The year movie was released")
    director: str=Field(description="Director of the movie")
    rating: float=Field(description="Rating of the movie out of 10")
    
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatOpenAI(output_version=None, profile={'name': 'GPT-5 Nano', 'release_date': '2025-08-07', 'last_updated': '2025-08-07', 'open_weights': False, 'max_input_tokens': 272000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001DF67DDC1A0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001DF67DDCC20>, root_client=<openai.OpenAI object at 0x000001DF67363620>, root_async_client=<openai.AsyncOpenAI object at 0x000001DF67DDC980>, model_name='gpt-5-n

In [4]:
response= model_with_structure.invoke("Provide details about the movie Inception")

response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

In [5]:

class Movie(BaseModel):
    title: str=Field(...,description="Title of the movie")
    year: int=Field(...,description="The year movie was released")
    director: str=Field(...,description="Director of the movie")
    rating: float=Field(...,description="Rating of the movie out of 10")
    
model_with_structure=model.with_structured_output(Movie, include_raw=True)
response= model_with_structure.invoke("Provide details about the movie Spider-Man")
response

{'raw': AIMessage(content='{"title":"Spider-Man","year":2002,"director":"Sam Raimi","rating":7.3}', additional_kwargs={'parsed': Movie(title='Spider-Man', year=2002, director='Sam Raimi', rating=7.3), 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1507, 'prompt_tokens': 112, 'total_tokens': 1619, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 1472, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DpTsOTdmf6CuxZGI9Gyf22033whKb', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019eb586-6b60-7ca2-8b00-f318871ed2f9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 112, 'output_tokens': 1507, 'total_tokens': 1619, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {

#### Nested Structure

In [6]:
class Actor(BaseModel):
    name: str
    role: str
    
class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in million USD")

model_with_structure=model.with_structured_output(MovieDetails)
response= model_with_structure.invoke("Provide details about the movie Spider-Man")
response

MovieDetails(title='Spider-Man', year=2002, cast=[Actor(name='Tobey Maguire', role='Peter Parker / Spider-Man'), Actor(name='Willem Dafoe', role='Norman Osborn / Green Goblin'), Actor(name='Kirsten Dunst', role='Mary Jane Watson'), Actor(name='James Franco', role='Harry Osborn'), Actor(name='Cliff Robertson', role='Uncle Ben'), Actor(name='Rosemary Harris', role='Aunt May'), Actor(name='J. K. Simmons', role='J. Jonah Jameson'), Actor(name='Bill Nunn', role='Robbie Robertson')], genres=['Action', 'Adventure', 'Sci-Fi'], budget=139.0)

#### TypedDict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [7]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details"""
    title: Annotated[str,...,"The title of the movie"]
    year: Annotated[int,...,"Yeat the movie was released"]
    
model_with_typedDict=model.with_structured_output(MovieDict)
response= model_with_typedDict.invoke("Provide details about the movie Spider-Man")
response

{'title': 'Spider-Man', 'year': 2002}

In [ ]:
class Actor(TypedDict):
    name: str
    role: str
    
class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in million USD")

model_with_structure=model.with_structured_output(MovieDetails)
response= model_with_structure.invoke("Provide details about the movie Spider-Man")
response

In [8]:
# model_with_structure.profile cant see profile for structured output

model.profile

{'name': 'GPT-5 Nano',
 'release_date': '2025-08-07',
 'last_updated': '2025-08-07',
 'open_weights': False,
 'max_input_tokens': 272000,
 'max_output_tokens': 128000,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': True,
 'temperature': False,
 'image_url_inputs': True,
 'pdf_inputs': True,
 'pdf_tool_message': True,
 'image_tool_message': True,
 'tool_choice': True}

#### DataClasses
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [9]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo 
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result)
print(result['structured_response'])

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='68fe11d5-5538-4b70-8da6-667049802036'), AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 612, 'prompt_tokens': 204, 'total_tokens': 816, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 576, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DpUKQZcpmSQGLQUfEHoyVHnHodcxC', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019eb5a0-ebe7-71e3-b231-51f0ede88105-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 204, 'output_t

In [10]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo 
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result)
print(result['structured_response'])

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='f1643365-b5c4-41f0-81c6-9bfa5e50686e'), AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 356, 'prompt_tokens': 204, 'total_tokens': 560, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DpUN2ulRrgGEfst919P99thiq4EkK', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019eb5a3-6654-77f2-9e60-fbee2a6291fc-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 204, 'output_t